# Sotiris code no squeezing

## Physical description

### Mechanics

The cells are spheroids that behave under the following equations:

$$\lambda_s \mathbf{v}_i  + \lambda_r \sum_j (\mathbf{v}_i - \mathbf{v}_j) = \sum_j \mathbf{F}_{ij}$$

where the force is

$$F_{ij}=
\begin{cases}
F_{att/rep}(\frac{r_{ij}}{d_{ij}}-1)(\frac{\mu r_{ij}}{d_{ij}}-1)\frac{(x_i-x_j)}{d_{ij}}\hspace{1cm}if\;d_{ij}<\mu r_{ij}\\
0\hspace{5cm}otherwise
\end{cases}$$

where $d_{ij}$ is the Euclidean distance and $r_{ij}$ is the sum of both radius.

### Biochemical interaction

The model considers 3 cellular states: A,B and C. Transitions are unidirectional from A to B and then B to C. The transition rate for a given cell to transition is given by:
$$\dot{\phi}_A = \frac{p}{1+\phi_A/K}$$
$$\dot{\phi}_B = \frac{q}{1+\phi_A/K}$$

where $\phi_A$ is the fraction of A-type cells in the system and $K$ is the feedback paramteter.


### Growth

The cells present division. The rules for the division in this model are. Random election of a division direction over the unit sphere. The daughter cells divide equally in mass and volume and are positioned in oposite directions around the division axis centered at the parent cell. A new division time is assigned to each aghter cell from a uniform distribution $\text{Uniform}(\tau_{div}(1-\sigma_{div}),\tau_{div}(1+\sigma_{div}))$.

## Packages

In [ ]:
using NBInclude
@nbinclude("preamble/packages.ipynb");
@nbinclude("preamble/functions.ipynb");

In [ ]:
using CellBasedModels
using Random
using Distributions
using CSV
using Pkg
using WriteVTK
using DataFrames, Distances, Clustering
using GeometryBasics
using Colors
using StaticArrays
using Statistics
using Dates

## Computational model

### Definition of the model

First, we have to create an instance of an agent with all the propoerties of the agents.First, we have to create an instance of an agent with all the propoerties of the agents.

In [ ]:
model = ABM(3,
   
# Global parameters
    model = Dict(

        :lambda => Float64,
        :f_range => Float64,
        :f_att => Array{Float64},
        :f_rep => Array{Float64},
        
        :kp_on => Array{Float64},             # Protrusion rate/"probability"
        :kp_off => Array{Float64},            # Protrusion duration rate/"probability"
        :prot_range => Float64,               # Protrusion force interaction range factor
        :prot_range_min => Float64,           # Protrusion force minimum range factor for establishing protrusion bond
        :prot_range_break => Float64,         # Protrusion force break range factor
        :p => Float64,                        # Transition rate from A to B
        :q => Float64,                        # Transition rate from B to C
        :feedback => Array{Float64},          # Transition rate feedback factor constant
        :prot_strength => Array{Float64},     # Contractile Protrusion force magnitude

    # Division constants:
        :t_div => Array{Float64},
        :t_div_pre => Array{Float64},
        :sigma_div => Array{Float64},

    # velocity dissipation:
        :vel_diss => Float64,
        
    # Physical constants
        :nbh_range => Float64,
        :r_i => Float64,   
    ),

    
# Local float parameters
    agent = Dict(

        :r => Float64,
        :vx => Float64,
        :vy => Float64,
        :vz => Float64,
        :fx => Float64,
        :fy => Float64,
        :fz => Float64,
        
        :fpx => Float64,
        :fpy => Float64,
        :fpz => Float64,
        
        :n_nbh => Int64,    # Number of neighbours that the agent has for velocity term in forces
        :n_nbh_a => Int64,  # Number of neighbours that the agent has in state A (cell_fate = 1)
        :n_nbh_b => Int64,  # Number of neighbours that the agent has in state B (cell_fate = 2)
        :n_nbh_c => Int64,  # Number of neighbours that the agent has in state C (cell_fate = 3)
        :n_nbh_prot => Int64,  # Number of neighbours that the agent has for Protrusion term in forces
        :nbh_vsum_x => Float64,   # Sum of neighbours' vx
        :nbh_vsum_y => Float64,   # Sum of neighbours' vy
        :nbh_vsum_z => Float64,   # Sum of neighbours' vz

        :kp_rng => Float64,       # Cell protrusion random number
        :prot_time_ct => Float64,       # Protrusion time constant
        :prot_time_ij => Float64,      # Protrusion time 
        :prot_timer => Float64,    # Protrusion time counter

    #For controlling division:
        :cell_dividing => Int64,
        :cell_div_relax_steps => Int64,

        :salt_and_pepper_switch => Int64,

        :N_start_sim_switch => Int64,

        :N_diff_min_switch => Int64,

        :prolif_active_switch => Int64,

        :N_active => Int64, # com.cell_fate components out of Nmax that are actually there!
        :N_state1 => Int64, # Number of com.cell_fate components that are = 1 (state A)!

        :prolif_active => Int64, # Tell if cell is chosen randomly for proliferation
        
        :division_time => Float64, #Variable storing the time of division of the cell
        :cell_fate => Int64 #Identity of the cell (1 A, 2 B, 3 C)
    ),


    
###Mechnical & Chemical dynamics
    agentODE = quote
        
        #Mechanics
        fx = 0.0; fy = 0.0; fz = 0.0
        fmax = 1000.0  # corresponds to a distance of R/8 between the cells
        switch_fmax_x = 0
        switch_fmax_y = 0
        switch_fmax_z = 0
        @loopOverNeighbors it2 begin
            dij = sqrt((x-x[it2])^2+(y-y[it2])^2+(z-z[it2])^2)
            rij = r+r[it2]
            
            if dij < f_range*rij && dij > 0
                
                if dij < rij

                    fx += ( f_rep[cell_fate[i1_],cell_fate[it2]] * (rij/dij-1)*(f_range*rij/dij-1) ) * (x-x[it2])/dij 
                    fy += ( f_rep[cell_fate[i1_],cell_fate[it2]] * (rij/dij-1)*(f_range*rij/dij-1) ) * (y-y[it2])/dij
                    fz += ( f_rep[cell_fate[i1_],cell_fate[it2]] * (rij/dij-1)*(f_range*rij/dij-1) ) * (z-z[it2])/dij

                    # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end
                
                else

                # Original force:
                    #1#:
                    fx += f_att[cell_fate[i1_],cell_fate[it2]]*(rij/dij-1)*(f_range*rij/dij-1)*(x-x[it2])/dij
                    fy += f_att[cell_fate[i1_],cell_fate[it2]]*(rij/dij-1)*(f_range*rij/dij-1)*(y-y[it2])/dij
                    fz += f_att[cell_fate[i1_],cell_fate[it2]]*(rij/dij-1)*(f_range*rij/dij-1)*(z-z[it2])/dij

                # softer force:
                    if fx > fmax
                        fx = fmax
                        if switch_fmax_x == 0
                            println(" **********   fx REACHED fmax!!!   **********")
                            switch_fmax_x = 1
                        end
                    end
                    if fy > fmax
                        fy = fmax
                        if switch_fmax_y == 0
                            println(" **********   fy REACHED fmax!!!   **********")
                            switch_fmax_y = 1
                        end
                    end
                    if fz > fmax
                        fz = fmax
                        if switch_fmax_z == 0
                            println(" **********   fz REACHED fmax!!!   **********")
                            switch_fmax_z = 1
                        end
                    end

                end
                
            end

            
        end
        
        if n_nbh < N_relV_min

            dt(x) = fx/lambda + fpx/lambda
            dt(y) = fy/lambda + fpy/lambda
            dt(z) = fz/lambda + fpz/lambda

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
           
        else
       
            dt(x) = fx/n_nbh/lambda + nbh_vsum_x/n_nbh + fpx/n_nbh/lambda
            dt(y) = fy/n_nbh/lambda + nbh_vsum_y/n_nbh + fpy/n_nbh/lambda
            dt(z) = fz/n_nbh/lambda + nbh_vsum_z/n_nbh + fpz/n_nbh/lambda 

            vx = dt(x)
            vy = dt(y)
            vz = dt(z)
            
        end
    

        compile=false
        #############################################################################################################################

    end,

    
    agentRule=quote

    # If we want to list the n_nbh:
        n_nbh_list = zeros(Int64, 0)
        
        n_nbh_prot_list = zeros(Int64, 0)

            
        fpx = 0.; fpy = 0.; fpz = 0.
    # Count neighbours for each cell & Add neighbour velocities for primary cell velocity calculation
        n_nbh_new = 0 #Set it to zero before starting the computation
        n_nbh_a_new = 0 #Set it to zero before starting the computation
        n_nbh_b_new = 0 #Set it to zero before starting the computation
        n_nbh_c_new = 0 #Set it to zero before starting the computation
        n_nbh_prot_new = 0 #Set it to zero before starting the computation
        nbh_vsum_x_new = 0. #Set it to zero before starting the computation
        nbh_vsum_y_new = 0. #Set it to zero before starting the computation
        nbh_vsum_z_new = 0. #Set it to zero before starting the computation
        nSumV_max = 100.0 #max vel force
        switch_nbh_vsum_x_max = 0
        switch_nbh_vsum_y_max = 0
        switch_nbh_vsum_z_max = 0
        halt_division = 0

        
        
    # FOR INITIALISING SALT & PEPPER DISTRIBUTION: 
        if N == N_start_sim && salt_and_pepper == 1 && salt_and_pepper_switch == 0

            if i1_ in selected_indices
                cell_fate = 2
            end
            salt_and_pepper_switch = 1
        end
        
        
        @loopOverNeighbors it2 begin
           
            d = CBMMetrics.euclidean(x,x[it2],y,y[it2],z,z[it2]) #Using euclidean matric provided in package
            ndist = (r + r[it2])*nbh_range
          
        # For Cell Velocity update:
            if d < ndist

                n_nbh_new += 1 #Add 1 to neighbors of cell

                if cell_fate[it2] == 1
                    n_nbh_a_new += 1 #Add 1 to A-neighbors of cell
                elseif cell_fate[it2] == 2
                    n_nbh_b_new += 1 #Add 1 to B-neighbors of cell
                elseif cell_fate[it2] == 3
                    n_nbh_c_new += 1 #Add 1 to C-neighbors of cell
                end

                if cell_dividing == 0 && cell_dividing[it2] == 0
                
                    nbh_vsum_x_new += vx[it2]*vel_diss
                    nbh_vsum_y_new += vy[it2]*vel_diss
                    nbh_vsum_z_new += vz[it2]*vel_diss
                        
                elseif cell_dividing > 0
                    
                    cell_dividing += 1
                    if cell_dividing > cell_div_relax_steps
                        cell_dividing = 0
                    end
                    
                end
                

            # If we want to list the n_nbh: 
                push!(n_nbh_list, it2)

            end

        # For Protrusion Pair Interaction:
            ndist_P = r*prot_range   # this is assuming all cells have the same radius! This gives FCC 3rd NNs! (Just before cells are allowed to pull other cells exactly behind their 1st NN (d=4r))
          
            if d < ndist_P && d > prot_range_min*r
                n_nbh_prot_new += 1

                push!(n_nbh_prot_list, it2)

            end
        
        end

        

        
    # For Protrusion Force Activation:

        if n_nbh_prot_new != 0
            random_neigh = Int64(ceil(CBMDistributions.uniform(0,1) * length(n_nbh_prot_list)))
            d = CBMMetrics.euclidean(x,x[random_neigh],y,y[random_neigh],z,z[random_neigh])
        end
        
        if n_nbh_prot_new != 0 && kp_rng < (kp_on[cell_fate[i1_],cell_fate[random_neigh]]*dt) && PI[i1_] == 0 && d < ndist_P && d >= prot_range_min*r && N >= N_prot_min && prot_ON == 1

            tij_new = prot_time_ct - log(CBMDistributions.uniform(0,1)) / kp_off[cell_fate[i1_],cell_fate[random_neigh]]

        # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            prot_time_ij = tij_new
            tij_c_new = 0.0
            prot_timer = tij_c_new
        
        # ALLOW ONE PROTRUSION PER CELL, BUT MORE THAN ONE BOND:
            PI[i1_] = random_neigh
            append!(PI_list[random_neigh], [i1_])
           
        else
            kp_rng_new = CBMDistributions.uniform(0,1)
            kp_rng = kp_rng_new
        end


    # Finally, update n_nbh and Neighbour Velocity Sums for cell:
        
        n_nbh = n_nbh_new
        n_nbh_a = n_nbh_a_new
        n_nbh_b = n_nbh_b_new
        n_nbh_c = n_nbh_c_new
        
        nbh_vsum_x = nbh_vsum_x_new
        nbh_vsum_y = nbh_vsum_y_new
        nbh_vsum_z = nbh_vsum_z_new

        n_nbh_prot = n_nbh_prot_new

        
    # Protrusion Force Profile:
       
        if (PI[i1_] != 0) && (N >= N_prot_min) && prot_ON == 1
           
            cell_partner = abs(PI[i1_])
           
            d_partner = CBMMetrics.euclidean(x,x[cell_partner],y,y[cell_partner],z,z[cell_partner])
            ndist_P = r * prot_range # Assuming FCC structure with all particles having same radius! 
            
            if prot_timer < prot_time_ij && d_partner < ndist_P && d_partner >= prot_range_break*r
            
                prot_timer += dt

                fpx -= prot_strength[cell_fate[i1_],cell_fate[cell_partner]] * (x-x[cell_partner])/d_partner
                fpy -= prot_strength[cell_fate[i1_],cell_fate[cell_partner]] * (y-y[cell_partner])/d_partner
                fpz -= prot_strength[cell_fate[i1_],cell_fate[cell_partner]] * (z-z[cell_partner])/d_partner
              
            elseif prot_timer >= prot_time_ij || d_partner >= ndist_P || d_partner < prot_range_break*r
            
                PI[i1_] = 0
                tij_c_new = 0.0
                prot_timer = tij_c_new
                kp_rng_new = CBMDistributions.uniform(0,1)
                kp_rng = kp_rng_new

                idx = findfirst(==(i1_), PI_list[cell_partner])
                if idx !== nothing
                    deleteat!(PI_list[cell_partner], idx)
                end
                
            end
        end

        
    # Compute force due to cell being bonded to other cells that extended prots to it:
        
        if (N >= N_prot_min) && prot_ON == 1
            for cell_partnered_to in PI_list[i1_]
                    
                d = CBMMetrics.euclidean(x,x[cell_partnered_to],y,y[cell_partnered_to],z,z[cell_partnered_to])
                ndist_P = r * prot_range # Assuming FCC structure with all particles having same radius!
        
                if prot_timer[cell_partnered_to] < prot_time_ij[cell_partnered_to] && d != 0 && d < ndist_P && d >= prot_range_break*r
                    
                    fpx -= prot_strength[cell_fate[cell_partnered_to],cell_fate[i1_]] * (x-x[cell_partnered_to])/d
                    fpy -= prot_strength[cell_fate[cell_partnered_to],cell_fate[i1_]] * (y-y[cell_partnered_to])/d
                    fpz -= prot_strength[cell_fate[cell_partnered_to],cell_fate[i1_]] * (z-z[cell_partnered_to])/d

                elseif prot_timer[cell_partnered_to] >= prot_time_ij[cell_partnered_to] || d >= ndist_P || d < prot_range_break*r

                    PI[cell_partnered_to] = 0
                    tij_c_new = 0.0
                    prot_timer[cell_partnered_to] = tij_c_new
                    kp_rng_new = CBMDistributions.uniform(0,1)
                    kp_rng[cell_partnered_to] = kp_rng_new

                    idx = findfirst(==(cell_partnered_to), PI_list[i1_])
                    if idx !== nothing
                        deleteat!(PI_list[i1_], idx)
                    end
                       
                end
                   
            end
        end
        #############################################################################################################################
       


        
    #Differentiation
        if N >= N_diff_min && diff_ON == 1

        # total active (non-zero) cells (Since I have Nmax slots, some will be 0 if N < Nmax !)
            N_active = count(!=(0), com.cell_fate)

        # number of cells in state 1
            N_state1 = count(==(1), com.cell_fate)
   

            if N >= N_start_sim && N_start_sim_switch == 0 && N_diff_min_switch < 2

               N_diff_min_switch += 1

            elseif  N >= N_start_sim && N_start_sim_switch == 1 && N_diff_min_switch < 2 && N == (N_start_sim + 50)

               N_diff_min_switch += 1
                
            end
            
            rand_number = CBMDistributions.uniform(0,1)
        
        # MEAN FIELD APPROACH:
            if n_nbh > 0 && cell_fate == 1 && rand_number < ( p*dt / (1.0 + feedback[cell_fate] * N_state1 / N_active) )
                cell_fate = 2
            elseif n_nbh > 0 && cell_fate == 2 && rand_number < ( q*dt / (1.0 + feedback[cell_fate] * N_state1 / N_active) )
                cell_fate = 3
            end
            
        end


        
    #Proliferation
        
        if N >= N_start_sim && N_start_sim_switch == 0
            τDiv_value = t_div[cell_fate]
            if N_start_sim_switch == 0
                division_time = t + CBMDistributions.uniform(τDiv_value*(1-sigma_div[cell_fate]),τDiv_value*(1+sigma_div[cell_fate]))
            end
            N_start_sim_switch = 1
        end


        if N_start_sim_switch == 0
            τDiv_value = t_div_pre[cell_fate]
        else
            τDiv_value = t_div[cell_fate]
        end



        
        
        if t > division_time && N < Nmax && prolif_active == 1
                
            if halt_division == 0
            
                if PI[i1_] != 0
                    cell_partner = abs(PI[i1_])
                    PI[i1_] = 0
                    PI[cell_partner] = 0
                    prot_timer = 0
                    prot_timer[cell_partner] = 0
                    kp_rng[cell_partner] = CBMDistributions.uniform(0,1)
                end
    
            #Choose random direction in unit sphere
                xₐ = CBMDistributions.normal(0,1); yₐ = CBMDistributions.normal(0,1); zₐ = CBMDistributions.normal(0,1)
                Tₐ = sqrt(xₐ^2+yₐ^2+zₐ^2)
                xₐ /= Tₐ; yₐ /= Tₐ; zₐ /= Tₐ    
    
            #Chose a random distribution of the material
                rnew = r
                rsep = 0.3*r 
                    
                @addAgent(          # add new agent
                    x = x+rsep*xₐ,
                    y = y+rsep*yₐ,
                    z = z+rsep*zₐ,
                
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
    
                    r = rnew,
                    division_time = t + CBMDistributions.uniform(τDiv_value*(1-sigma_div[cell_fate]),τDiv_value*(1+sigma_div[cell_fate])),
                    kp_rng = CBMDistributions.uniform(0,1),
                    prot_time_ij = 0.0,
                    cell_dividing = 1,
                )
                @addAgent(          # add new agent
                    x = x-rsep*xₐ,
                    y = y-rsep*yₐ,
                    z = z-rsep*zₐ,
                   
                    vx = vx/2.,
                    vy = vy/2.,
                    vz = vz/2.,
                    
                    r = rnew,
                    division_time = t + CBMDistributions.uniform(τDiv_value*(1-sigma_div[cell_fate]),τDiv_value*(1+sigma_div[cell_fate])),
                    kp_rng = CBMDistributions.uniform(0,1),
                    prot_time_ij = 0.0,
                    cell_dividing = 1,
                )
                @removeAgent()      # remove agent that divided
            
            else

                halt_division = 0
                
            end

        end

    end,

    agentAlg=CBMIntegrators.Heun()
);

### Community construction and initialisation

Once with the model created, we have to construct an initial Community of agents to evolve.

#### Parameters

The model from the original version has some parameters defined. We create a dictionary with all the parameters from the model assigned.

In [ ]:
parameters = Dict([

    :nbh_range  =>  1.0,
    :r_i  =>  1.0,
    :prot_range  =>  2.0,
    :prot_range_min  =>  2.0,
    :prot_range_break  =>  1.0,
    :lambda  =>  1E+0,   
    :kp_on  =>  [0.0 0.0 0.0; 1.0 1.0 1.0; 5.0 5.0 5.0],
    :kp_off  =>  [200.0 200.0 200.0; 2.0 0.1 2.0; 2.0 2.0 2.0],
    :prot_time_ct  =>  0.0,
    :p  =>  0.009333,
    :q  =>  0.004667,
    :feedback  =>  [5.0 5.0 0.0],
    :vel_diss  =>  0.9, # velocity/momentum dissipation for relative velocity
    :f_range  =>  2.0, #this must be greater than 1 for the model to make sense!
    :t_div_pre  =>  [5 5 5],
    :t_div  =>  [300 300 750],
    :sigma_div  =>  [0.5 0.5 0.5],
    :f_att  =>  [6.0 6.0 2.4; 6.0 6.0 1.6; 2.4 1.6 4.8],
    :f_rep  =>  [3.0 3.0 3.0; 3.0 3.0 3.0; 3.0 3.0 5.1],
    :prot_strength  =>  [5 2 5; 2 5 2; 5 7 7], # protrusion forces
        
]);

#### Initialise the community

The model starts from just one agent. Create the community and assign all the parameters to the Community object.

In [ ]:
function initializeEmbryo(parameters;dt,N)

    com = Community(
                model,
                N=N,
                dt=dt,
                )

    #Global parameters
    for (par,val) in pairs(parameters)
        com[par] = val
    end
    
    #Initialise locals
    com.r = parameters[:r_i]
    com.f_att = parameters[:f_att]
    com.cell_fate = 1 #Start neutral (A) fate 
    com.cell_dividing = 0
    com.cell_div_relax_steps = cell_div_relax_steps
    com.salt_and_pepper_switch = 0
    com.N_start_sim_switch = 0
    com.N_diff_min_switch = 0
    com.prolif_active_switch = 0
    com.N_state1 = 1 # should be 1
    com.N_active = 1 # should be 1
    com.prolif_active = 1
   
    #Initialise variables
    #############################################################################################################################
    com.n_nbh = 0 #Start with the n_nbh = 0
    com.n_nbh_a = 0 #Start with the n_nbh_a = 0
    com.n_nbh_b = 0 #Start with the n_nbh_b = 0
    com.n_nbh_c = 0 #Start with the n_nbh_c = 0
    com.n_nbh_prot = 0 #Start with the n_nbh_prot = 0
    com.nbh_vsum_x = 0. #Start with the nbh_vsum_x = 0.
    com.nbh_vsum_y = 0. #Start with the nbh_vsum_y = 0.
    com.nbh_vsum_z = 0. #Start with the nbh_vsum_z = 0.
    com.prot_time_ij = 0. #Start with prot_time_ij = 0.
    com.prot_timer = 0. #Start with prot_time_ij = 0.
    com.kp_rng = CBMDistributions.uniform(0,1) #Start with random vlaues of kp_rng drawn from uniform dist.
    #############################################################################################################################
    com.x = 0.
    com.y =  0.
    com.z =  0.
    com.vx = 0.
    com.vy = 0.
    com.vz = 0.
    com.division_time = 1 #rand(Uniform(com.τDiv-com.σDiv,com.τDiv+com.σDiv))

    return com

end;

### Creating a custom evolve step

In [ ]:
function customEvolve!(com,steps,saveEach)
    loadToPlatform!(com,preallocateAgents = Nmax-N_ini+1) #loadToPlatform!(com,preallocateAgents = 100)
    switch_0 = 0
    switch_1 = 0
    switch_2 = 0
    for i in 1:steps

        if i % 1000 == 0
            println("      ***** i = ", i, " / ", steps, " and com = " , com.N, " agents *****    \n")
            switch_1 = 0
            switch_2 = 0            
        end

        

        agentStepDE!(com)
        agentStepRule!(com)
        update!(com)
        computeNeighbors!(com)
        
        if i % saveEach == 0
            saveRAM!(com)
        end
        
        #Stop by time
        if (switch_0 == 0) && all(com.N .>= Nmax)  # if all(com.N .> 60)
            println("\n\n\n --------- REACHED THRESHHOLD! com.N = ",com.N, " REACHED THRESHHOLD! ---------")
            switch_0 = 1
        end
        
    end
    bringFromPlatform!(com)
    
end;

## Simulate

We check how the agents starts to divide and choose a fate at late stages of the simulation.

In [ ]:
random_seed = 74375
Random.seed!(random_seed)

rng = MersenneTwister(random_seed)  # Create an explicit RNG instance with the seed


dt = 0.02 #0.001 #0.005 #0.0005  ##0.0002  ###0.005   #small early stage tests: 0.0005

# Number of cells at start of simulation
N_ini = 1 # minimum value of 1!

# Maximum number of cells
# Nmax = 15000
Nmax = 1000

# Number of cells after which real simulation starts:
# N_start_sim = 300
N_start_sim = 100

# Active cell-cell protrusions
prot_ON = 0

# When protrusions kick in
N_prot_min = N_start_sim

# Cell differentiation
diff_ON = 1
N_diff_min = N_start_sim

# Relative velocities
N_relV_min = 1  #When relative friction kicks in.
cell_div_relax_steps = 150




# TO CREATE SALT & PEPPER DISTRIBUTION OF CELL FATES (ONLY WORKS WHEN STARTING WITH 100% A-TYPE AGGREGATES!):

salt_and_pepper = 0
S_and_P_B_proportion = 0.0


if salt_and_pepper == 1
    
    # How many of those to convert?
    num_to_convert = round(Int, N_start_sim * S_and_P_B_proportion)
    
    # Select which ones to convert randomly
    selected_indices = randperm(rng, N_start_sim)[1:num_to_convert]  # Random indices from type 1
    
end


PI = zeros(Int64, Nmax)

# If we want to list the n_nbh:
n_nbh_list = zeros(Int64, 0)
n_nbh_prot_list = zeros(Int64, 0)


# If we want to know which cells are bonded by prots to chosen cell:
PI_list = [Int64[] for _ in 1:Nmax]


steps = round(Int64,200/dt)
saveEach = round(Int64,1/dt)


com = initializeEmbryo(parameters,dt=dt,N=N_ini);
customEvolve!(com,steps,saveEach)

println(" ")
println("Finished simulation. ",com)


In [ ]:
function plot_aggregate(com, color_map, mstart, mstop;
	size = ((maximum(com.x) - minimum(com.x)) + com.r[1]) / 1.5,
	n = 4,
    showtime = false,
    shownumbers = true
)

	fig = Figure(resolution = (n * 640, 600), figure_padding = 40)
	labelsize = 60
	d = getParameter(com, [:x, :y, :z, :r, :cell_fate])

	for (i, pos) in enumerate(range(start = mstart, length = n, stop = mstop))
		pos = floor(Int, pos)
		println("Plot $i: timestamp $pos")
        t = round(com[pos].t, digits=2)
		ax = Axis3(
            fig[1, i],
            aspect = :data,
            xlabel = "",
            ylabel = "",
            zlabel = "",
			xticklabelsize = labelsize,
			yticklabelsize = labelsize,
			zticklabelsize = labelsize,
            titlevisible = showtime,
            titlealign = :center,
            titlegap = 12,
            titlesize = labelsize,
            title = L"t=%$(t)"
        )
        if !shownumbers
            ax.xticklabelsize = 0
			ax.yticklabelsize = 0
			ax.zticklabelsize = 0
        end
		color = [color_map[j] for j in d[:cell_fate][pos]]
		meshscatter!(ax, d[:x][pos], d[:y][pos], d[:z][pos], markersize = d[:r][pos], color = color) # d[:r][pos] will be constant = r
		xlims!(ax, -size, size)
		ylims!(ax, -size, size)
		zlims!(ax, -size, size)
	end

	display(fig)

end;

In [ ]:
plot_aggregate(com, color_map, 1, length(com))

## Save cell positions and fates with time



### Make directory for saving output files 



In [ ]:
dir_name = "results"
mkpath(dir_name)

In [ ]:
d = getParameter(com,[:x,:y,:z,:cell_fate]);
df = DataFrame(d);

In [ ]:
using DataFrames

times_expanded = Int[]
x_expanded = Float64[]
y_expanded = Float64[]
z_expanded = Float64[]
fate_expanded = Int[]

for (i, row) in enumerate(eachrow(df))  # i is timestep index
    n_cells = length(row.x)
    
    append!(times_expanded, fill(i, n_cells))  # use i as time
    
    append!(x_expanded, row.x)
    append!(y_expanded, row.y)
    append!(z_expanded, row.z)
    append!(fate_expanded, row.cell_fate)
end

flat_df = DataFrame(t = times_expanded,
                    x = x_expanded,
                    y = y_expanded,
                    z = z_expanded,
                    cell_fate = fate_expanded)

# Now save as CSV
using CSV
CSV.write("$dir_name/cells.csv", flat_df)